In [ ]:
import pandas as pd
import numpy as np
import math
import heapq
import distance_metrics

In [ ]:
df = pd.read_csv('Iris.csv')
x = df.iloc[:, 1:5]
y = df.iloc[:, 5:]

rows, cols = x.shape

# Shuffle and split
shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x.iloc[train_idx].values
x_test = x.iloc[test_idx].values
y_train = y.iloc[train_idx, 0].values
y_test = y.iloc[test_idx, 0].values

print(f"Training samples: {len(x_train)}, Test samples: {len(x_test)}")

In [ ]:
#Setting an odd numbered K
k = 5
#This predicts an example by simply choosing the k nearest vectors
#from the training examples using the Euclidean Distance and
#then taking the majiority of its labels as the prediction
def predict_one(testing_vector, training_examples, labels, dist_func):

    #Storing all K neighbours in a heap
    heap = []
    for i in range(len(training_examples)):
        dist = dist_func(training_examples[i], testing_vector)
        #Pushing if the heap capacity isn't k
        if len(heap) < k:
            heapq.heappush_max(heap, (dist, i, labels[i]))
        #Popping out the vector with the largest distance so
        #a vector with a smaller distance can be added
        elif dist < heap[0][0]:
            heapq.heappop_max(heap)
            heapq.heappush_max(heap, (dist, i, labels[i]))
    
    #Creating a counter for the occurence of each label in the
    #K nearest vectors (neighbours)
    votes = {}
    for i in range(len(heap)):
        if heap[i][2] not in votes:
            votes[heap[i][2]] = 1
        else:
            votes[heap[i][2]] += 1

    sorted_map = sorted(votes.items(), key=lambda item:item[1], reverse=True)

    return next(iter(sorted_map))[0]

#Gives an accuracy for all from the testing examples
def predict_all(testing_examples, training_examples, training_labels, testing_labels):
    accuracy = 0
    for i in range(len(testing_examples)):
        if(predict_one(testing_examples[i], training_examples, training_labels, dist_func=distance_metrics.Euclidean_dist) == testing_labels[i]):
            accuracy += 1
    
    return accuracy/len(testing_examples)

predict_all(x_test, x_train, y_train, y_test)
